# Individual edits

Each dermatologist gets their own codebook. Reliability, text generality, and image generality run on the edited cases. Locality is fourth: cases no user marked wrong, scored before retrieval against after retrieval.

`dermedit` kernel. `OVERWRITE` recomputes saved runs. `RESUME` finishes a run that is still missing a metric. With both off, a complete result is printed from disk and an unfinished one stays blank.

In [ ]:
import sys
from pathlib import Path

PERTURB = Path("/scratch/jq2uw/MME/derm_vlm_edit/editing/perturb_generality")
sys.path.insert(0, str(PERTURB))
from utils.bootstrap import bootstrap, results_dir
bootstrap()

import config as C

USERS = [
    "jesse.y.liu.md@gmail.com",
    "achiou@stanford.edu",
    "lhunterellul",
]
MODELS = ["medgemma", "dermatollama"]
RATIONALE_MODE = C.RATIONALE_MODE
NORM_ANSWER = "y16"
SUBSAMPLE = 0
MAX_NEW_TOKENS = C.MAX_NEW_TOKENS
SEED = 42
OVERWRITE = False
RESUME = True

# EMBED_MODE:
#   standalone_dual  DINOv2 image concatenated with SBERT text. This is the current run.
#   dual_sbert       VLM vision activation concatenated with SBERT. Still dual.
#   vision           VLM vision activation only.
#   language         A middle language-layer activation only.
#   language_last    Late language-layer activation. The old VLM-key style.
EMBED_MODE = "standalone_dual"
# Percentile of key-key distances. 1 and 5 are the sweep. 0.1 is the renamed run.
REJECT_THRESHOLD = 1
IMAGE_ENCODER = "dinov2"
IMAGE_POOL = "cls"
TEXT_ENCODER = "sbert"
SCALER_DATASET = "coco"

OUT = results_dir(PERTURB, "individual", str(EMBED_MODE) + "_" + str(REJECT_THRESHOLD))
print("results:", OUT)

In [ ]:
import gc

from IPython.display import Markdown, display

from utils.edit import edit
from utils.edits import collect_locality, collect_user
from utils.eval import load_suite, load_vlm, needs_compute, render_table, run_suite

rows = []
for model_short in MODELS:
    cached = {}
    todo = []
    for user in USERS:
        if needs_compute(OUT, user, model_short, overwrite=OVERWRITE, resume=RESUME):
            todo.append(user)
        else:
            print(f"{user} / {model_short}: loaded")
            cached[user] = load_suite(OUT, user, model_short)
    if todo:
        import torch
        editor = None
        locality = collect_locality(USERS, model_short, subsample=SUBSAMPLE)
        vlm, config = load_vlm(
            model_short, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
            embed_mode=EMBED_MODE, image_encoder=IMAGE_ENCODER,
            image_pool=IMAGE_POOL, text_encoder=TEXT_ENCODER,
            scaler_dataset=SCALER_DATASET, reject_threshold=REJECT_THRESHOLD,
        )
        for user in todo:
            edits, _ = collect_user(
                user, model_short, rationale_mode=RATIONALE_MODE,
                norm_answer=NORM_ANSWER, subsample=SUBSAMPLE,
            )
            if not edits:
                print(f"skip {user} / {model_short}")
                continue
            editor, queries = edit(vlm, config, edits)
            cached[user] = run_suite(
                editor, vlm, queries, user=user, model_short=model_short,
                out_dir=OUT, locality_edits=locality, seed=SEED,
                norm_answer=NORM_ANSWER, max_new_tokens=MAX_NEW_TOKENS,
                overwrite=OVERWRITE,
            )
        del vlm, editor
        gc.collect()
        torch.cuda.empty_cache()
    for user in USERS:
        if user in cached:
            rows.append(cached[user])

display(Markdown(render_table(rows)))